# 1-Year Total Energy Consumption: Conventional Intersection vs. STI Systems

Horizontal stacked-bar figure comparing a conventional signalized-intersection baseline against Basic, Semi-Automated, and Highly-Automated STI stacks. Style matches the companion CAV figure.

In [ ]:
# Cell 1 — imports, figure config, title config, font config, color map
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

FIG_TITLE = '1-Year Total Energy Consumption: Conventional Intersection vs. STI Systems'
X_LABEL = 'Annual Energy Consumption (kWh)'
FIG_SIZE = (14, 8)

COLORS = {
    'Baseline Signal Control': '#D3D3D3',
    'Sensing':                 '#33697B',
    'Communication':           '#FFAD9E',
    'Computing':               '#ABBDC5',
}

COMPONENT_ORDER = ['Baseline Signal Control', 'Sensing', 'Communication', 'Computing']

In [ ]:
# Cell 2 — hard-coded baseline data + STI subsystem data

# Conventional signalized-intersection baseline (replaces propulsion in the CAV plot).
# Average ~1510 kWh/yr (~4.14 kWh/day measured); upper reference ~3888 kWh/yr
# (~324 kWh/month measured).
Baseline_Avg = 1510.0
Baseline_Min = 1510.0
Baseline_Max = 3888.0

# STI autonomy subsystem annual energy (kWh/yr), base case.
sti_energy_1yr = {
    'Basic STI': {
        'Sensing':       176.08,
        'Communication': 569.40,
        'Computing':     5308.46,
    },
    'Semi-Automated STI': {
        'Sensing':       1054.70,
        'Communication': 735.84,
        'Computing':     21233.83,
    },
    'Highly-Automated STI': {
        'Sensing':       1303.49,
        'Communication': 884.76,
        'Computing':     42467.65,
    },
}

# Display order — top to bottom (matplotlib barh draws the last row at the top,
# so we build the DataFrame in this order and reverse at plot time).
LEVEL_ORDER = ['Basic STI', 'Semi-Automated STI', 'Highly-Automated STI']

In [ ]:
# Cell 3 — dataframe construction, totals, percentages, error-bar extents

rows = []
for label in LEVEL_ORDER:
    sub = sti_energy_1yr[label]
    sensing = sub['Sensing']
    comm    = sub['Communication']
    comp    = sub['Computing']
    sti_sys = sensing + comm + comp
    total   = Baseline_Avg + sti_sys
    rows.append({
        'Label':                   label,
        'Baseline Signal Control': Baseline_Avg,
        'Baseline_Min':            Baseline_Min,
        'Baseline_Max':            Baseline_Max,
        'Sensing':                 sensing,
        'Communication':           comm,
        'Computing':               comp,
        'Total':                   total,
        'STI_Sys':                 sti_sys,
        'STI_Sys_Pct':             100.0 * sti_sys / total,
        'Error_Min':               total - (Baseline_Avg - Baseline_Min),
        'Error_Max':               total + (Baseline_Max - Baseline_Avg),
    })

df = pd.DataFrame(rows, columns=[
    'Label', 'Baseline Signal Control', 'Baseline_Min', 'Baseline_Max',
    'Sensing', 'Communication', 'Computing',
    'Total', 'STI_Sys', 'STI_Sys_Pct', 'Error_Min', 'Error_Max',
])

print(df.to_string(index=False))

In [ ]:
# Cell 4 — plotting, annotations, legend, save, show

fig, ax = plt.subplots(figsize=FIG_SIZE)

# barh draws the first y-position at the bottom; reverse so the top bar is the
# first level in LEVEL_ORDER.
plot_df = df.iloc[::-1].reset_index(drop=True)
y_pos = np.arange(len(plot_df))
bar_height = 0.5

# Horizontal stacked bars — stack order left→right matches COMPONENT_ORDER.
left = np.zeros(len(plot_df))
for comp in COMPONENT_ORDER:
    ax.barh(y_pos, plot_df[comp].values, left=left,
            height=bar_height, color=COLORS[comp],
            edgecolor='white', linewidth=0.6, label=comp)
    left = left + plot_df[comp].values

# Black horizontal error bars centered at Total.
totals = plot_df['Total'].values
err_lo = totals - plot_df['Error_Min'].values
err_hi = plot_df['Error_Max'].values - totals
ax.errorbar(totals, y_pos, xerr=[err_lo, err_hi], fmt='none',
            ecolor='black', elinewidth=1.2, capsize=5, capthick=1.2, zorder=5)

# Right-side text block per bar.
x_max_data = plot_df['Error_Max'].max()
text_offset = x_max_data * 0.015
for i, row in plot_df.iterrows():
    x_text = row['Error_Max'] + text_offset
    ax.text(x_text, y_pos[i] + 0.12,
            f"Total: {row['Total']:,.0f} kWh",
            va='center', ha='left', fontsize=11, fontweight='bold',
            color='#222222')
    breakdown = (
        f"Base: {row['Baseline Signal Control']:,.0f} kWh\n"
        f"Comp: {row['Computing']:,.0f} kWh\n"
        f"Sens: {row['Sensing']:,.0f} kWh\n"
        f"Comm: {row['Communication']:,.0f} kWh\n"
        f"STI Sys%: {row['STI_Sys_Pct']:.1f}%"
    )
    ax.text(x_text, y_pos[i] - 0.08,
            breakdown,
            va='top', ha='left', fontsize=9,
            color='#555555', linespacing=1.25)

# Axis formatting.
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df['Label'].values)
ax.set_xlabel(X_LABEL)
ax.set_title(FIG_TITLE, loc='left', pad=12)

# x-axis margin so the right-side text block has room.
ax.set_xlim(0, x_max_data * 1.32)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _p: f"{int(v):,}"))

# Extra y-axis padding below the last bar so the lower-right legend sits in
# clean whitespace and does not collide with the breakdown text.
ax.set_ylim(-1.1, len(plot_df) - 0.4)

# Remove top/right/left spines; keep bottom only.
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.tick_params(axis='x', length=4, color='#333333')

# Legend ordered top→bottom: Computing, Communication, Sensing, Baseline Signal Control.
legend_order = ['Computing', 'Communication', 'Sensing', 'Baseline Signal Control']
legend_handles = [Patch(facecolor=COLORS[k], edgecolor='white', label=k)
                  for k in legend_order]
ax.legend(handles=legend_handles, loc='lower right', frameon=False)

plt.tight_layout()

fig.savefig('sti_1yr_total_energy_consumption.png', dpi=300, bbox_inches='tight')
fig.savefig('sti_1yr_total_energy_consumption.pdf', bbox_inches='tight')

plt.show()

In [ ]:
# Cell 5 — summary print

print('STI System Proportion for each level (as percentage of total):')
print()
for _, row in df.iterrows():
    print(f"{row['Label']}:")
    print(f"  Total:      {row['Total']:,.0f} kWh")
    print(f"  Base:       {row['Baseline Signal Control']:,.0f} kWh")
    print(f"  STI Sys:    {row['STI_Sys']:,.0f} kWh")
    print(f"  STI Sys%:   {row['STI_Sys_Pct']:.1f}%")
    print()